In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from datetime import datetime

from utils.data_loader import create_datasets
from utils.models import build_baseline_cnn

In [ ]:
model_name = 'RAF'

Load pretrained model to fine-tune it? If yes, specify model name, if no set it to ```None```.

In [ ]:
pretrained_model_name = None

# Data

## Preprocessing Function
Preprocessing function, where pixel values per channel are normalized to have zero mean and unit variance. Mean and standard-deviation are obtained from the training set only.

In [ ]:
# RAF-DB training data mean and std
mean = [146.6770, 114.6274, 102.3102]
std = [67.6282, 61.7651, 61.3665]

def preprocess(x):
    # ensure image format
    x = np.array(x, dtype='float32')
    
    # normalize
    x[..., 0] -= mean[0]
    x[..., 1] -= mean[1]
    x[..., 2] -= mean[2]
    if std is not None:
        x[..., 0] /= std[0]
        x[..., 1] /= std[1]
        x[..., 2] /= std[2] 
    return x

def de_preprocess(x):
    if std is not None:
        x[..., 0] *= std[0]
        x[..., 1] *= std[1]
        x[..., 2] *= std[2]
    # normalize
    x[..., 0] += mean[0]
    x[..., 1] += mean[1]
    x[..., 2] += mean[2]
    return x.astype('uint8')

## Load Data

In [ ]:
%%time

# settings
IMG_SIZE = (100, 100)
IMG_SHAPE = IMG_SIZE + (3,)
BATCH_SIZE = 16 # Nên để 16 để khớp với máy cá nhân
# load data
train_ds, val_ds, test_ds, class_weights, info = create_datasets(
    data_dir='data/DATASET',
    img_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    val_split=0.2,
    augment_train=True
)

Prepare some function to get labels...

In [ ]:
emotion_labels = ['surprise', 'fear', 'disgust', 'happiness', 'sadness', 'anger', 'neutral']

num_classes = len(emotion_labels)
print("Number of classes: ", num_classes)

def vec2label(onehot_vec):
    major_vote = np.argmax(onehot_vec)
    return emotion_labels[major_vote]

## Data Examples

Show some examples...

In [ ]:
images, labels = next(iter(train_ds))
images = images.numpy()
plt.subplots(4,4, figsize=(10, 10))
for i in range(16):
    plt.subplot(4,4, i+1)
    plt.imshow(
        de_preprocess(images[i])
    )
    plt.title(vec2label(labels[i]))
    plt.axis('off')
plt.show()

# Model

## Base model
Architecture of the model without classification layer.

In [ ]:
model = build_baseline_cnn(input_shape=IMG_SHAPE, num_classes=num_classes)

model.summary()


## Classification model
Add classification layer to the base model.

In [ ]:
model.add(tf.keras.layers.Dense(num_classes, activation='softmax', name="softmax"))

# load pretrained version if required
if pretrained_model_name is not None:
    model.load_weights(f'./models/{pretrained_model_name}')

# compile the model
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate = 0.001),
              loss='sparse_categorical_crossentropy',  # <--- Updated line
              metrics=['accuracy'])

model.summary()


# Training

In [ ]:
# callbacks
dt = datetime.now().strftime("%m%d-%H%M")

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(filepath=f"./modelcheckpoints/{model_name}_{dt}.keras",
                                       monitor='val_accuracy',
                                       save_best_only=True,
                                       verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss',
                                         factor=0.1,
                                         patience=4,
                                         verbose=1,
                                         mode='auto',
                                         min_lr=0.00000001),
    tf.keras.callbacks.CSVLogger(f'./log/{model_name}_{dt}.csv',
                                 separator=",", append=True)
]

In [ ]:
import os
os.makedirs('./log', exist_ok=True)
os.makedirs('./modelcheckpoints', exist_ok=True)
os.makedirs('./outputs/models', exist_ok=True)


In [ ]:
%%time

epochs = 20

history = model.fit(train_ds,
                    epochs=epochs,
                    validation_data=val_ds,
                    class_weight=class_weights,
                    callbacks=callbacks)


# Results

## Training Results

In [ ]:
plt.figure(figsize=(12, 4))
# Accuracy
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training')
plt.plot(history.history['val_accuracy'], label='Validation')
plt.title("Accuracy")
plt.grid(); plt.legend()
# Loss
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training')
plt.plot(history.history['val_loss'], label='Validation')
plt.title("Loss")
plt.grid(); plt.legend()
plt.show()

## Test Results

In [ ]:
loss, acc = model.evaluate(test_ds, verbose=2)

print("\nAccuracy:\t%.2f%%" % (acc * 100))
print("Loss:\t\t%.4f" % (loss))

# Save Model

In [ ]:
import os

# Lưu thẳng vào thư mục outputs/models để file webcam tự đọc được
os.makedirs('outputs/models', exist_ok=True)
model.save_weights('outputs/models/baseline_cnn.keras')

print(" Đã lưu model thành công! Bạn đã có thể bật Webcam lên chạy!")


---